In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

import lightgbm as lgb

In [2]:
SEEDS = [42, 2024, 777]
N_FOLDS = 10

In [3]:
train = pd.read_csv('/kaggle/input/first-competition-exhibition/train.csv')
test  = pd.read_csv('/kaggle/input/first-competition-exhibition/test.csv')

test_ids = test['id']

train = train.drop(columns=['id', 'Row#'])
test  = test.drop(columns=['id', 'Row#'])


In [4]:
def create_features(df, kmeans=None, scaler=None, fit=False):
    data = df.copy()

    # ---- Bees ----
    data['total_bees'] = (
        data['honeybee'] +
        data['bumbles'] +
        data['andrena'] +
        data['osmia']
    )

    data['bees_per_clone'] = data['total_bees'] / (data['clonesize'] + 1e-6)

    data['osmia_honeybee']   = data['osmia'] * data['honeybee']
    data['bumble_honeybee'] = data['bumbles'] * data['honeybee']
    data['andrena_osmia']   = data['andrena'] * data['osmia']

    # ---- Temperature ----
    data['temp_range'] = data['MaxOfUpperTRange'] - data['MinOfLowerTRange']

    data['avg_temp'] = (
        data['AverageOfUpperTRange'] +
        data['AverageOfLowerTRange']
    ) / 2

    data['temp_x_rain'] = data['avg_temp'] * data['RainingDays']
    data['temp_rain_ratio'] = data['avg_temp'] / (data['RainingDays'] + 1e-6)

    # ---- Statistics ----
    bee_cols = ['honeybee','bumbles','andrena','osmia']
    data['bees_mean'] = data[bee_cols].mean(axis=1)
    data['bees_std']  = data[bee_cols].std(axis=1)

    # ---- Non-linear (اصلاح‌شده) ----
    for col in ['clonesize', 'total_bees', 'fruitmass', 'seeds']:
        data[f'log_{col}'] = np.log1p(data[col])

    data['fruitmass_sq'] = data['fruitmass'] ** 2
    data['seeds_sq']     = data['seeds'] ** 2

    data['fruit_seed_ratio'] = data['fruitmass'] / (data['seeds'] + 1e-6)

    # ---- Clustering ----
    cluster_cols = [
        'clonesize',
        'total_bees',
        'avg_temp',
        'RainingDays',
        'fruitmass'
    ]

    if fit:
        scaler = StandardScaler()
        scaled = scaler.fit_transform(data[cluster_cols])

        kmeans = KMeans(
            n_clusters=8,
            random_state=42,
            n_init=30
        )
        data['cluster'] = kmeans.fit_predict(scaled)
    else:
        scaled = scaler.transform(data[cluster_cols])
        data['cluster'] = kmeans.predict(scaled)

    data['cluster_bees'] = data['cluster'] * data['total_bees']

    return data, kmeans, scaler


# =====================================================
# Feature creation
# =====================================================
train_fe, kmeans, scaler = create_features(train, fit=True)
test_fe, _, _ = create_features(test, kmeans=kmeans, scaler=scaler)

X = train_fe.drop(columns=['yield'])
y = np.log1p(train_fe['yield'])

In [5]:
BASE_PARAMS = {
    'objective': 'huber',
    'alpha': 0.9,
    'metric': 'mae',

    'learning_rate': 0.02,
    'num_leaves': 128,
    'min_data_in_leaf': 20,

    'feature_fraction': 0.9,
    'bagging_fraction': 0.9,
    'bagging_freq': 1,

    'lambda_l1': 0.5,
    'lambda_l2': 0.5,
    'min_gain_to_split': 0.01,

    'boosting': 'gbdt',

    'device': 'gpu',
    'gpu_platform_id': 0,
    'gpu_device_id': 0,
    'max_bin': 255,

    'verbosity': -1
}

# =====================================================
# Multi-seed CV + Ensemble
# =====================================================
final_test_preds = np.zeros(len(test_fe))
all_oof = []

for seed in SEEDS:
    print(f'\n===== SEED {seed} =====')

    params = BASE_PARAMS.copy()
    params['seed'] = seed

    kf = KFold(
        n_splits=N_FOLDS,
        shuffle=True,
        random_state=seed
    )

    oof = np.zeros(len(X))
    test_preds = np.zeros(len(test_fe))
    maes = []

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X), 1):
        print(f'Fold {fold}/{N_FOLDS}')

        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        train_set = lgb.Dataset(X_tr, y_tr)
        val_set   = lgb.Dataset(X_val, y_val)

        model = lgb.train(
            params,
            train_set,
            num_boost_round=8000,
            valid_sets=[val_set],
            callbacks=[
                lgb.early_stopping(300, verbose=False),
                lgb.log_evaluation(0)
            ]
        )

        val_pred = np.expm1(model.predict(X_val))
        oof[val_idx] = val_pred

        fold_mae = mean_absolute_error(
            np.expm1(y_val),
            val_pred
        )
        maes.append(fold_mae)

        test_preds += np.expm1(model.predict(test_fe)) / N_FOLDS

    print(f'SEED {seed} OOF MAE:', np.mean(maes))
    all_oof.append(oof)
    final_test_preds += test_preds / len(SEEDS)



===== SEED 42 =====
Fold 1/10


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


Fold 2/10
Fold 3/10
Fold 4/10
Fold 5/10
Fold 6/10
Fold 7/10
Fold 8/10
Fold 9/10
Fold 10/10
SEED 42 OOF MAE: 248.2178145387217

===== SEED 2024 =====
Fold 1/10
Fold 2/10
Fold 3/10
Fold 4/10
Fold 5/10
Fold 6/10
Fold 7/10
Fold 8/10
Fold 9/10
Fold 10/10
SEED 2024 OOF MAE: 247.5102398535095

===== SEED 777 =====
Fold 1/10
Fold 2/10
Fold 3/10
Fold 4/10
Fold 5/10
Fold 6/10
Fold 7/10
Fold 8/10
Fold 9/10
Fold 10/10
SEED 777 OOF MAE: 247.87855577057334


In [6]:
final_test_preds = np.clip(
    final_test_preds,
    train['yield'].min(),
    train['yield'].max()
)

submission = pd.DataFrame({
    'id': test_ids,
    'yield': final_test_preds
})

submission.to_csv('submission.csv', index=False)
submission.head()

,id,yield
0,15000,7451.234820
1,15001,5820.483183
2,15002,6672.396183
3,15003,4643.307460
4,15004,5866.844694
